In [ ]:
!pip install PyMuPDF

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.1/24.1 MB 58.4 MB/s eta 0:00:00


In [ ]:
import fitz  # PyMuPDF
import os
import re

def clean_text(text):
    """Applies basic cleaning rules to the extracted text."""
    text = text.replace('-\n', '')
    text = re.sub(r'(?<!\n)\n(?!\n)', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

# Get a list of all files in the current directory
all_files = os.listdir('.')
# Filter for PDF files
pdf_files = [f for f in all_files if f.lower().endswith(".pdf")]

print(f"Found {len(pdf_files)} PDF(s) to process.")

# Process each PDF
for pdf_filename in pdf_files:
    print(f"--- Processing: {pdf_filename} ---")
    try:
        doc = fitz.open(pdf_filename)
        full_raw_text = ""
        for page in doc:
            full_raw_text += page.get_text()
        doc.close()

        cleaned_text = clean_text(full_raw_text)

        # Create the output filename
        txt_filename = os.path.splitext(pdf_filename)[0] + ".txt"

        # Save the cleaned text to a new file
        with open(txt_filename, "w", encoding="utf-8") as f:
            f.write(cleaned_text)

        print(f"✅ Successfully extracted and saved to: {txt_filename}")

    except Exception as e:
        print(f"❌ ERROR processing {pdf_filename}: {e}")

Found 1 PDF(s) to process.
--- Processing: PathologyRobbins7ed.pdf ---
✅ Successfully extracted and saved to: PathologyRobbins7ed.txt


In [ ]:
!pip install langchain langchain_community langchain-google-genai sentence-transformers faiss-cpu

In [ ]:
import os
from langchain_community.document_loaders import TextLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# 1. Find and load your documents
txt_files = [f for f in os.listdir('.') if f.lower().endswith(".txt")]

# --- ADDED FIX ---
# Check if any .txt files were found before proceeding
if not txt_files:
    print("❌ ERROR: No .txt files found.")
    print("Please re-run the PDF extraction script from the earlier step to create the text files.")
else:
    print(f"Found {len(txt_files)} text file(s) to load.")
    loaders = [TextLoader(file, encoding='utf-8') for file in txt_files]
    documents = []
    for loader in loaders:
        documents.extend(loader.load())

    # 2. Chunk the documents
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)
    docs = text_splitter.split_documents(documents)
    print(f"Split documents into {len(docs)} chunks.")

    # --- ADDED FIX ---
    # Check if there are chunks to add to the database
    if not docs:
        print("❌ ERROR: The text files were loaded but resulted in zero chunks. The files might be empty.")
    else:
        # 3. Create embeddings and store in a FAISS vector database
        print("Creating embeddings... (This may take a moment)")
        embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
        db = FAISS.from_documents(docs, embeddings) # This line is now safe

        print("✅ Vector database created successfully!")
        db.save_local("my_faiss_index")

Found 1 text file(s) to load.
Split documents into 2682 chunks.
Creating embeddings... (This may take a moment)
✅ Vector database created successfully!


In [ ]:
db.save_local("my_faiss_index")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# The path "/content/drive/My Drive/" points to the main folder of your Google Drive
permanent_save_path = "/content/drive/My Drive/my_faiss_index"

# Save the database to the new path
db.save_local(permanent_save_path)

print(f"Successfully saved the index permanently to your Google Drive at: {permanent_save_path}")

Successfully saved the index permanently to your Google Drive at: /content/drive/My Drive/my_faiss_index
